# 🔁 Cuaderno 04 — Matrices especiales y métodos iterativos

**Métodos Numéricos · Unidad 3: Sistemas de Ecuaciones Lineales**

Este cuaderno es **taller, no libro**. Toca dos familias de métodos que existen porque la matriz tiene **estructura** que se puede aprovechar:

- **Matrices especiales:** tridiagonales (algoritmo de Thomas) y simétricas definidas positivas (Cholesky).
- **Métodos iterativos:** Jacobi y Gauss-Seidel, que en vez de transformar la matriz se van acercando a la solución.

📖 Teoría completa:
[`03_metodos_iterativos.md`](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/03_metodos_iterativos.md) ·
[`04_matrices_especiales.md`](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/04_matrices_especiales.md)

### 🎯 Lo que vas a entregar

- **Pregunta 9** — `mi_jacobi_paso(A, b, x)`: **una** barrida de Jacobi.
- **Pregunta 10** — `mi_gauss_seidel_paso(A, b, x)`: **una** barrida de Gauss-Seidel.
- **Pregunta 11** — `mi_thomas(e, f, g, r)`: el sistema tridiagonal completo.

### ⚙️ ¿Por qué una sola barrida y no el método completo?

Porque el método completo depende del **criterio de paro**, que es una convención (¿tolerancia de $10^{-6}$? ¿error absoluto o relativo?). Una barrida es un resultado **exacto** que se puede comprobar sin ambigüedad. Y es la parte donde de verdad se aprende la diferencia entre Jacobi y Gauss-Seidel.

Todo con aritmética exacta (`M.sym`): verás $151/48$, no `3.1458333333333335`.


In [ ]:
#@title ⚙️ Cargar entorno estilo MATLAB (ejecuta PRIMERO)

import os, sys, subprocess, importlib

# --- Detecta si corremos en Google Colab ---
try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

REPO_URL = "https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales.git"


def _buscar_raiz(base, saltos=6):
    """Sube desde `base` hasta la carpeta que tiene config/config.yaml."""
    b = base
    for _ in range(saltos):
        if os.path.exists(os.path.join(b, "config", "config.yaml")):
            return os.path.abspath(b)
        padre = os.path.dirname(b)
        if padre == b:
            break
        b = padre
    return None


if ES_COLAB:
    # ============================================================
    # Google Colab: montar Drive y traer el repositorio
    # ============================================================
    # El repo se clona DENTRO de tu Drive, no en /content (que se borra al
    # cerrar la sesión). Así sobrevive entre sesiones y `git pull` solo
    # actualiza lo que cambió.
    from google.colab import drive

    drive.mount("/content/drive")

    BASE = "/content/drive/MyDrive/CursoMN"
    REPO = os.path.join(BASE, "U3_Repo")
    os.makedirs(BASE, exist_ok=True)

    try:
        if not os.path.exists(os.path.join(REPO, ".git")):
            print("Clonando el repositorio (la primera vez tarda un poco)...")
            subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
        else:
            print("Actualizando el repositorio...")
            subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
        print("✅ Repositorio listo en", REPO)
    except Exception as _exc:
        print("⚠️ No pude clonar/actualizar el repositorio:", _exc)
        print("   Si la carpeta ya existe en tu Drive, puedes continuar.")
else:
    # En tu computadora el repo ya está en disco: solo hay que encontrarlo.
    _ruta_nb = globals().get("__vsc_ipynb_file__")      # VS Code
    REPO = _buscar_raiz(os.path.dirname(os.path.abspath(_ruta_nb))) if _ruta_nb else None
    REPO = REPO or _buscar_raiz(os.getcwd())

if REPO is None:
    raise RuntimeError(
        "No encuentro la carpeta del repositorio (la que tiene config/config.yaml).\n"
        "Abre el cuaderno desde dentro de U3_Repo o revisa que el clon de Colab exista."
    )

print("Repositorio :", REPO)

# ============================================================
# Biblioteca del curso (lib/robomat.py)
# ============================================================
# En sys.path van las DOS raíces que hacen falta:
#   REPO/lib    -> robomat.py y el paquete mn_motor/
#   REPO        -> el paquete preguntas/ (las preguntas de este cuaderno)
for _ruta in (os.path.join(REPO, "lib"), REPO):
    if _ruta not in sys.path:
        sys.path.insert(0, _ruta)

import robomat as _robomat
importlib.reload(_robomat)          # recoge los cambios si editaste la libreria
from robomat import *               # noqa: F401,F403

print("✅ Entorno cargado desde:", _robomat.__file__)


# ============================================================
# Motor de calificación
# ============================================================
# En Colab SIEMPRE se usa el blob ofuscado. En tu computadora puedes escoger
# "modulos": importa lib/mn_motor/ tal cual, y así funcionan F12, el
# autocompletado y los puntos de ruptura. Es el modo para depurar.
MODO_MOTOR = "modulos"

MODO = "blob" if ES_COLAB else MODO_MOTOR

if MODO == "modulos":
    try:
        # Se borran del caché para que una edición en lib/mn_motor/ o en
        # preguntas/ se note al volver a correr la celda, sin reiniciar.
        for _n in [n for n in sys.modules
                   if n.split(".")[0] in ("mn_motor", "preguntas")]:
            del sys.modules[_n]
        # Import ESTÁTICO después del borrado: es lo que Pylance analiza.
        import mn_motor as _motor
        from mn_motor import *        # noqa: F401,F403
    except ImportError as _exc:
        # En un clon del alumno no existe preguntas/: no hay clave de
        # respuestas. Sin problema: el blob trae lo mismo.
        print("ℹ️ No pude importar el motor del repositorio:", _exc)
        print("   Uso el motor ofuscado del blob.")
        MODO = "blob"

if MODO == "blob":
    import base64 as _b64
    import zlib as _zlib

    # Cada cuaderno tiene su propio blob: aquí va el del Cuaderno 04.
    ruta_motor = os.path.join(REPO, "grader", "u3_04_especiales_iterativos.txt")
    with open(ruta_motor, encoding="utf-8") as _f:
        _blob = _f.read().strip()
    _codigo = _zlib.decompress(_b64.b64decode(_blob)).decode("utf-8")
    # _REPO_U3 le dice al motor dónde está el repo: dentro del blob no existe
    # __file__, así que no puede deducirlo solo.
    os.environ["U3_REPO"] = REPO
    _MOTOR = {"__name__": "blob_u3"}
    exec(_codigo, _MOTOR)
    globals().update({_n: _MOTOR[_n] for _n in _MOTOR["__all__"]})


# Este cuaderno califica SU conjunto de preguntas.
# En modo blob ya viene incrustado, pero en modo módulos hay que declararlo.
usar_cuaderno("u3_04_especiales_iterativos", "U3_04_Especiales_Iterativos")

print(f"✅ Motor de calificación cargado ({MODO}) · cuaderno {cuaderno()}")
print("   Sigue con la celda de identificación.")


## 0 · Tu identificación

Todo lo que sigue usa **tus** datos. Al ejecutar la celda de abajo, Colab te
pide permiso para leer tu correo; de ahí salen tu número de control y tu nombre.
Con eso se siembran los números de tus preguntas: **cada alumno recibe datos
distintos**.

Fuera de Colab no hay sesión de Google, así que el correo se escribe a mano en
la celda.


In [ ]:
#@title 👤 Identificación automática (correo institucional)
# ============================================================
# Uso el correo con el que montaste Drive: de ahí salen tu NC y tu nombre.
# ============================================================
if ES_COLAB:
    import requests
    import google.auth
    import google.auth.transport.requests
    from google.colab import auth

    try:
        auth.authenticate_user()
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        info = requests.get(
            "https://www.googleapis.com/drive/v3/about?fields=user",
            headers={"Authorization": "Bearer " + creds.token},
            timeout=20,
        ).json()
        cuenta = info.get("user", {})
        email = cuenta.get("emailAddress", "")
        nombre = cuenta.get("displayName", "")
    except Exception as e:
        print("⚠️ No pude obtener tu correo automáticamente.")
        print("   Vuelve a ejecutar y acepta el permiso en la ventana de Google.")
        print("   Detalle:", e)
        email, nombre = "", ""
else:
    # Modo local: aquí no hay sesión de Google, escribe tu correo.
    email = "m16330887@hermosillo.tecnm.mx"      # <-- cambia por el tuyo
    nombre = ""                                 # Google solo lo da en Colab

if email and email.lower().endswith(".tecnm.mx"):
    alumno_id = email
    print("✅ Identificado:", email)
    print("   Número de control:", extraer_nc(email))
    print("   Nombre:", nombre if nombre else "(no disponible fuera de Colab)")
else:
    from IPython.display import HTML, display as _display

    alumno_id = None
    _display(HTML(
        '<p style="color:#b00020;background:#ffe0e0;padding:12px;border-radius:8px;'
        'font-weight:bold">⛔ Necesito tu correo institucional (@….tecnm.mx).<br>'
        'Si iniciaste sesión con otra cuenta, ciérrala o usa una ventana de '
        'incógnito.</p>'))
    print("Correo detectado:", repr(email))

if alumno_id is None:
    raise RuntimeError("Detente: sin correo institucional no puedo generar tu tarea.")

# La tarea se genera UNA sola vez: todas las celdas de preguntas llaman a
# pregunta(n) sobre este MI_TAREA, así que todas hablan de lo mismo.
MI_TAREA = generar_tarea(alumno_id, nombre)
print(f"\nListo: {len(MI_TAREA['preguntas'])} preguntas para el NC {MI_TAREA['nc']}.")


## 1 · El mapa del cuaderno

Todo lo anterior (Gauss, LU) es **directo**: se transforma la matriz y en un
número finito y predecible de pasos sale la solución. Lo que viene es distinto.

**Métodos iterativos.** Se parte de un vector inicial $\mathbf{x}^{(0)}$ y se
genera una sucesión que *debería* acercarse a la solución:

$$x_i = \frac{b_i - \sum_{j \neq i} a_{ij}x_j}{a_{ii}}$$

Es decir: de cada ecuación se despeja **su propia** incógnita. La ventaja aparece
con matrices grandes y **dispersas** (casi todo ceros), donde eliminar sería un
desperdicio. La desventaja: la convergencia no está garantizada.

**Matrices especiales.** Si la matriz tiene ceros en casi todos lados, hay
algoritmos que nunca guardan esos ceros: el de **Thomas** para tridiagonales
([sección 4.1](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/04_matrices_especiales.md#sec-4-1)) y **Cholesky** para simétricas definidas positivas ([sección 4.2](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/04_matrices_especiales.md#sec-4-2)).

El sistema iterativo de todo el cuaderno, cuya solución es
$\mathbf{x}=[1,\ 2,\ 3]$:


In [ ]:
#@title El sistema iterativo del cuaderno
A = M.sym[[4, 1, -1], [2, 7, 1], [1, -3, 12]]
b = M.sym[[3], [19], [31]]

mostrar(A)
mostrar(b)
print("solución esperada:  x1 = 1,  x2 = 2,  x3 = 3")
print("¿exacto?", es_exacto(A))


## 2 · La condición que hace que funcione

Nada garantiza que iterar converja. La garantía más usada es la **dominancia
diagonal estricta por renglones**: en cada ecuación, el coeficiente de su propia
incógnita pesa más que la suma de los demás.

$$|a_{ii}| > \sum_{j \neq i} |a_{ij}| \qquad \text{para todo } i$$

Si se cumple, Gauss-Seidel (y Jacobi) convergen a la solución única **para
cualquier** $\mathbf{x}^{(0)}$. No es una condición necesaria, pero sí es
suficiente y se verifica antes de programar. Está en la
[sección 3.3](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/03_metodos_iterativos.md#sec-3-3).

Compruébalo abajo con dos matrices: la del cuaderno y un contraejemplo.


In [ ]:
#@title ✅ Comprobar la dominancia diagonal
def es_dominante(Mx):
    """¿Se cumple |a_ii| > suma de los demás |a_ij| en todos los renglones?"""
    n = Mx.rows
    for i in range(n):
        resto = sum(abs(Mx[i, j]) for j in range(n) if j != i)
        print(f"  renglón {i + 1}: |{Mx[i, i]}| > {resto}  ->  {abs(Mx[i, i]) > resto}")
        if not abs(Mx[i, i]) > resto:
            return False
    return True


print("La matriz del cuaderno:")
mostrar(A)
print("¿dominante?", es_dominante(A))

print("\nUn contraejemplo:")
C = M.sym[[1, 3], [4, 1]]
mostrar(C)
print("¿dominante?", es_dominante(C))


## 3 · Jacobi: una barrida

Una **barrida** (o iteración) calcula todas las incógnitas nuevas a partir de las
viejas:

$$x_i^{(k+1)} = \frac{1}{a_{ii}}\left(b_i - \sum_{j \neq i} a_{ij}\,x_j^{(k)}\right)$$

```
nuevo = lista de n ceros
para i = 1, ..., n:
    suma = 0
    para j = 1, ..., n, j != i:
        suma = suma + A[i][j] * x[j]        # x es el vector VIEJO
    nuevo[i] = (b[i] - suma) / A[i][i]
devolver nuevo
```

La clave está en la última columna del pseudocódigo: se lee de `x`, el vector
viejo, y se escribe en `nuevo`. Las incógnitas **no se ven entre sí** dentro de
la misma barrida.

> 💡 En la pregunta 9 el vector inicial **no** es cero. Si escribes el código
> arrancando de ceros y te olvidas de usar `x`, pasarás el ejemplo del cuaderno
> y fallarás los tres casos.


In [ ]:
#@title ✍️ Escribe una barrida de Jacobi
def mi_jacobi_paso(A, b, x):
    """Una barrida de Jacobi: todo con el vector viejo x."""
    raise NotImplementedError("Traduce aquí el pseudocódigo de la sección 3")


practica('prog_jacobi')


In [ ]:
#@title 📝 Pregunta 9 (vale 1 punto)
pregunta(9)


## 4 · Gauss-Seidel: la misma cuenta, un detalle

Gauss-Seidel **reutiliza de inmediato** lo que acaba de calcular:

$$x_i^{(k+1)} = \frac{1}{a_{ii}}\left(b_i - \sum_{j<i} a_{ij}\,x_j^{(k+1)} - \sum_{j>i} a_{ij}\,x_j^{(k)}\right)$$

Mirando los superíndices: las incógnitas **anteriores** ($j<i$) ya están en la
iteración nueva, las **posteriores** ($j>i$) siguen viejas.

```
para i = 1, ..., n:
    suma = 0
    para j = 1, ..., n, j != i:
        suma = suma + A[i][j] * x[j]        # x se va actualizando
    x[i] = (b[i] - suma) / A[i][i]
devolver x
```

La única diferencia con Jacobi es que **hay una sola lista**: se sobrescribe
`x[i]` en el momento, y el renglón siguiente ya ve el valor nuevo. Esa
sustitución inmediata es todo el algoritmo, y por eso suele converger más rápido.

En la celda de abajo se ven las dos primeras barridas lado a lado, con el mismo
punto de partida $\mathbf{x}^{(0)} = [0,0,0]$.


In [ ]:
#@title ✍️ Escribe una barrida de Gauss-Seidel
def mi_gauss_seidel_paso(A, b, x):
    """Una barrida de Gauss-Seidel: sobrescribe x al vuelo."""
    raise NotImplementedError("Traduce aquí el pseudocódigo de la sección 4")


practica('prog_seidel')


In [ ]:
#@title 🔍 Jacobi y Gauss-Seidel, lado a lado (con el mismo arranque)
try:
    j1 = mi_jacobi_paso(A, b, [0, 0, 0])
    s1 = mi_gauss_seidel_paso(A, b, [0, 0, 0])

    print("Barrida 1 con Jacobi (todo viejo):")
    mostrar(j1)
    print("Barrida 1 con Gauss-Seidel (reutiliza):")
    mostrar(s1)

    print("\nDiferencia (Gauss-Seidel menos Jacobi):")
    mostrar(s1 - j1)
    print("Gauss-Seidel se acerca más rápido, y hace exactamente el mismo número de cuentas.")
except (NotImplementedError, NameError):
    print("⏳ Escribe primero las dos funciones (secciones 3 y 4).")


In [ ]:
#@title 📝 Pregunta 10 (vale 1 punto)
pregunta(10)


## 5 · Matrices tridiagonales: el algoritmo de Thomas

Una matriz **tridiagonal** solo tiene coeficientes en la diagonal ($f$) y en las
dos diagonales vecinas ($e$ abajo, $g$ arriba):

$$\begin{bmatrix} f_1 & g_1 & 0 & 0 \\ e_2 & f_2 & g_2 & 0 \\ 0 & e_3 & f_3 & g_3 \\ 0 & 0 & e_4 & f_4 \end{bmatrix}$$

Thomas es la eliminación gaussiana **sin pivoteo** sobre esa estructura, en tres
fases:

```
# fase 1: modificar f  (k = 2..n)
e'[k] = e[k] / f'[k-1]
f'[k] = f[k] - e'[k] * g[k-1]

# fase 2: modificar r  (k = 2..n)
r'[k] = r[k] - e'[k] * r'[k-1]

# fase 3: sustitución hacia atrás  (k = n..1)
x[n] = r'[n] / f'[n]
x[k] = (r'[k] - g[k] * x[k+1]) / f'[k]
```

En la **primera** fila no hay nada que modificar: `f'[1] = f[1]` y `r'[1] = r[1]`.
Ese es el error más común al programarlo.

**Convención de los vectores** (la del documento y la que usa la pregunta 11):
los cuatro vectores miden $n$, y las posiciones que no existen valen 0.
`e[i]` multiplica a $x_{i-1}$, así que `e[0]` no se usa; `g[i]` multiplica a
$x_{i+1}$, así que `g[n-1]` no se usa.

> 💡 La ganancia: $O(n)$ en tiempo y memoria, contra $O(n^3)$ y $O(n^2)$ de la
> eliminación normal. No guarda ni un cero.


In [ ]:
#@title ✍️ Escribe el algoritmo de Thomas
# Ejemplo del documento (solución [1, 1, 1]): los cuatro vectores miden 3.
e_ej = [0, -1, -1]      # e[0] no se usa
f_ej = [2, 2, 2]
g_ej = [-1, -1, 0]      # g[n-1] no se usa
r_ej = [1, 0, 1]


def mi_thomas(e, f, g, r):
    """Resuelve el sistema tridiagonal y devuelve x como columna."""
    raise NotImplementedError("Traduce aquí las tres fases de la sección 5")


practica('prog_thomas')


In [ ]:
#@title 📝 Pregunta 11 (vale 1 punto)
pregunta(11)


## 6 · Cholesky: una sola triangular

Si $\mathbf{A}$ es **simétrica** ($a_{ij} = a_{ji}$) y **definida positiva**
($\mathbf{x}^{T}\mathbf{A}\mathbf{x} > 0$ para todo $\mathbf{x}\neq\mathbf{0}$),
la factorización se reduce a una sola matriz:

$$\mathbf{A} = \mathbf{L}\mathbf{L}^{T}$$

Las recurrencias, renglón por renglón, son

$$l_{kk} = \sqrt{a_{kk} - \sum_{j<k} l_{kj}^{2}}, \qquad l_{ki} = \frac{a_{ki} - \sum_{j<i} l_{ij}l_{kj}}{l_{ii}} \;\;(i<k)$$

Se calcula primero la **raíz de la diagonal** y luego se dividen las entradas del
renglón. Tres consecuencias: cuesta $\tfrac{1}{3}n^3$ (la mitad que LU), solo se
guarda la mitad inferior y **no necesita pivoteo**, porque el radicando siempre
es positivo.

La celda de abajo la calcula con la matriz del documento y comprueba que
$\mathbf{L}\mathbf{L}^{T} = \mathbf{A}$.


In [ ]:
#@title 🧪 Cholesky sobre la matriz del documento
Ac = M.sym[[4, 12, -16], [12, 37, -43], [-16, -43, 98]]
mostrar(Ac)
print("¿simétrica?", Ac == Ac.T)

n = Ac.rows
Lc = zeros_sym(n, n)
for k in range(n):
    radicando = Ac[k, k] - sum(Lc[k, j] ** 2 for j in range(k))
    Lc[k, k] = sqrt(radicando)              # sqrt exacta: deja sqrt(2) si hace falta
    for i in range(k + 1, n):
        Lc[i, k] = (Ac[i, k] - sum(Lc[i, j] * Lc[k, j] for j in range(k))) / Lc[k, k]

print("L =")
mostrar(Lc)
print("¿L @ Lᵀ = A?", (Lc @ Lc.T) == Ac)
print("¿exacto?", es_exacto(Lc))


## 7 · Preguntas de teoría (1 a 8)

Las cinco primeras son de opción múltiple. La 6 pide una matriz (armar la
tridiagonal), la 7 un pivote modificado de Thomas y la 8 una entrada de la
diagonal de Cholesky.


In [ ]:
#@title 🧠 Preguntas 1 a 5 · opción múltiple
pregunta(1)
pregunta(2)
pregunta(3)
pregunta(4)
pregunta(5)


In [ ]:
#@title ✍️ Preguntas 6 a 8 · con tus matrices
# La 6 es una matriz (escríbela como [[1,2,3,4],[5,6,7,8],...]);
# la 7 y la 8 son un solo número.
pregunta(6)
pregunta(7)
pregunta(8)


## 8 · Calificar y enviar

- `calificar()` **no envía nada**: úsala las veces que quieras mientras trabajas. En las de programar te dice **qué caso** falló.
- `enviar(alumno_id)` sí manda la calificación y **gasta uno de tus 5 intentos**.


In [ ]:
#@title 📝 Calificar (sin enviar) — úsala las veces que quieras
calificar()


In [ ]:
#@title 📤 Enviar la calificación (máximo 5 intentos)
enviar(alumno_id)


In [ ]:
#@title 🔎 Comprobar lo que ya está guardado (no envía nada)
consultar_calificacion()


---
### ✅ Con esto cierras la Unidad 3

Ya tienes los cuatro pilares: Gauss con pivoteo, LU, los métodos iterativos y los
algoritmos para matrices con estructura. La comparación entre todos —cuál
conviene en cada caso— está en
[`05_comparativa.md`](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/05_comparativa.md).

📤 Antes de cerrar: ejecuta `calificar()` y luego `enviar(alumno_id)`. Recuerda
que tienes **5 intentos**.
